<h2>Before you start</h2>
If this is the first time the pipeline is running on this machine, just run the cell below. It will copy startup.py from the BEARMIND folder into your local startup folder. This allows the code in startup.py to be executed automatically after each kernel restart (and removes the need to monotonously click through all setup cells after each reloading).

In [2]:
import shutil
import os

local_startup_dir = get_ipython().profile_dir.startup_dir
filedir = os.getcwd()
shutil.copy(os.path.join(filedir, 'startup.py'), os.path.join(local_startup_dir, 'startup.py'))

'/Users/nikita/.ipython/profile_default/startup/startup.py'

<h2>Module 0</h2>
You need to specify the root folder and pathway pattern. Note that * is a wildcard for any symbol combination except slashes (i.e., for any folder name), so it is strongly recommended to use it here.<br/><br/>
NB!! Just in case, use double backslashes for folder separation, otherwise some symbols may be interpreted as escape sequences. 

In [ ]:
config_data = {
    'ROOT': "C:\\Users\\admin\\YandexDisk\\_Projects\\FOF\\CalciumData\\4_Estimates\\",
    #'DATA_PATHWAY': 'legacy',
    'DATA_PATHWAY': 'bonsai',
    'TEMP_PATHWAY': "c:\\Users\\1\\caiman_data\\temp\\"
}

update_config(config_data)

In [ ]:
CONFIG

In [ ]:
create_mouse_configs(root=CONFIG['ROOT'])
create_session_configs(root=CONFIG['ROOT'])

<h2>Module 1</h2>
Manual video inspection. <br/>Open folder with miniscopic videos in a pop-up window, wait for loading and specify margins to be cropped by sliders or by keyboard, then save them by running the next cell. At the time, cropping .pickle files are to be created in these folders. Repeat for all folders with miniscopic videos you would like to analyze.   

In [ ]:
#Manual file selection:
fnames = list(askopenfilenames(title = 'Select files for inspection', initialdir = CONFIG['ROOT'], filetypes = [('AVI files', '.avi')]))

data = LoadSelectedVideos(fnames)
w = DrawCropper(data, fname=fnames[0])

Batch cropping and timestamp extraction.<br/>Miniscopic videos from folders with .pickle files are to be cropped and saved as _CR.tif in the root folder. There is no need for renaming of sigle-digit .avi files (like 0-9.avi to 00-09.avi)!<br/>
Also, along with video data, timestamps are to be copied from minicopic folders to the root folder. Do not delete them, they are nessesary for the further steps!

## Combined Modules 1-2-2.5

In [ ]:
#Batch crop
cpath_template = os.path.normpath(os.path.join(CONFIG['ROOT'], folder_structure, '*cropping.pickle'))
pick_names = glob(cpath_template)

print([get_session_name_from_path(fname) for fname in pick_names])
# TODO: read from mouse or sconfig only!

for name in pick_names:
    DoCropAndRewrite(name, sort = True, write_mp4 = True)
    extract_and_copy_ts(name)

#Automatic file selection
fnames = glob(os.path.join(CONFIG['ROOT'], '*_CR.tif'))
#OR, alternatively, you can use manual file selection:
#fnames = askopenfilenames(title = 'Select files for motion correction', initialdir = CONFIG['ROOT'], filetypes = [('TIFF files', '.tif')])

mc_dict = {
    'pw_rigid': False,         # flag for performing piecewise-rigid motion correction (otherwise just rigid)
    'max_shifts': (35, 35),    # maximum allowed rigid shift
    'gSig_filt': (8, 8),       # size of high pass spatial filtering, used in 1p data
    'strides': (48, 48),       # start a new patch for pw-rigid motion correction every x pixels
    'overlaps': (24, 24),      # overlap between pathes (size of patch strides+overlaps)
    'max_deviation_rigid': 15,  # maximum deviation allowed for patch with respect to rigid shifts
    'border_nan': 'copy',      # replicate values along the boundaries
    'use_cuda': True,          # Set to True in order to use GPU
    'memory_fact': CONFIG['RAM']/16.0,          # How much memory to allocate. 1 works for 16Gb, so 0.8 showd be optimized for 12Gb.
    'niter_rig': 1,
    'splits_rig': 20,          # for parallelization split the movies in  num_splits chuncks across time
                               # if none all the splits are processed and the movie is saved
    'num_splits_to_process_rig': None,
    'write_mp4': True} # intervals at which patches are laid out for motion correction  

for name in tqdm.tqdm(fnames):
    DoMotionCorrection(name, mc_dict)
    temp_pathway = CONFIG['TEMP_PATHWAY']
    sub_name = '/'.join(name.split('/')[6:])
    memmap_name = f'{temp_pathway}/{sub_name}'
    CleanMemmaps(memmap_name)

#%matplotlib ipympl
fnames = glob(os.path.join(CONFIG['ROOT'], '*_MC.tif'))
#fnames = askopenfilenames(title = 'Select files for corr image testing',
#                          initialdir = CONFIG['ROOT'],
#                          filetypes = [('TIFF files', '.tif')])

plot_gsig_range(fnames, maxframes=10000, min_gsig=3, max_gsig=6, step=5, dpi=300,
                    show_images=0, save_images=1)
plot_min_corr_and_pnr_range(fnames, maxframes=2000,
                            gsig_range=[3,4,5], pnr_range=[5,7,10],
                            mincorr_range=[0.85, 0.9, 0.95],
                            step=5, dpi=300,
                            show_images=0, save_images=1)

<h2>Module 2</h2>
Batch motion correction.<br/>All _CR.tif files in the root folder are to be automatically motion corrected with NoRMCorre routine [Pnevmatikakis, Giovanucci, 2017] with the parameters below and saved as _MC.tif files.

In [ ]:
#Automatic file selection
#fnames = glob(os.path.join(CONFIG['ROOT'], '*_CR.tif'))
#OR, alternatively, you can use manual file selection:
fnames = askopenfilenames(title = 'Select files for motion correction', initialdir = CONFIG['ROOT'], filetypes = [('TIFF files', '.tif')])

mc_dict = {
    'pw_rigid': False,         # flag for performing piecewise-rigid motion correction (otherwise just rigid)
    'max_shifts': (35, 35),    # maximum allowed rigid shift
    'gSig_filt': (8, 8),       # size of high pass spatial filtering, used in 1p data
    'strides': (48, 48),       # start a new patch for pw-rigid motion correction every x pixels
    'overlaps': (24, 24),      # overlap between pathes (size of patch strides+overlaps)
    'max_deviation_rigid': 15,  # maximum deviation allowed for patch with respect to rigid shifts
    'border_nan': 'copy',      # replicate values along the boundaries
    'use_cuda': True,          # Set to True in order to use GPU
    'memory_fact': CONFIG['RAM']/16.0,          # How much memory to allocate. 1 works for 16Gb, so 0.8 showd be optimized for 12Gb.
    'niter_rig': 1,
    'splits_rig': 20,          # for parallelization split the movies in  num_splits chuncks across time
                               # if none all the splits are processed and the movie is saved
    'num_splits_to_process_rig': None} # intervals at which patches are laid out for motion correction  

for name in tqdm.tqdm(fnames):
    DoMotionCorrection(name, mc_dict)
    CleanMemmaps(name)

In [ ]:
ms_name = 'H02'
session_name = 'NOF_H02_0D'
mc_to_config = {'mc_params': mc_dict}

ms_config_path = get_mouse_config_path(ms_name)
session_config_path = get_session_config_path(session_name)

#update_config(mc_to_config, cpath=ms_config_path)
update_config(mc_to_config, cpath=session_config_path)

<h3>Module 2.5 (optional)</h3>
Pre-test of various values of <i>gSig</i> parameter, which is used in the Module 3 and corresponds to a typical radius of a neuron in pixels.<br/>You can play with this parameter but you can use the default value of gSig = 6 as well. <br/> Calculation may take a while, so be patient!

Code snippet for manual calculation of imax for corrupted images

In [ ]:
#%matplotlib ipympl
fnames = glob(os.path.join(CONFIG['ROOT'], '*_MC.tif'))
#fnames = askopenfilenames(title = 'Select files for corr image testing',
#                          initialdir = CONFIG['ROOT'],
#                          filetypes = [('TIFF files', '.tif')])

plot_gsig_range(fnames, maxframes=10000, min_gsig=3, max_gsig=6, step=5, dpi=300,
                    show_images=0, save_images=1)
plot_min_corr_and_pnr_range(fnames, maxframes=2000,
                            gsig_range=[3,4,5], pnr_range=[5,7,10],
                            mincorr_range=[0.85, 0.9, 0.95],
                            step=5, dpi=300,
                            show_images=0, save_images=1)

In [ ]:
'''session_name = ['RFC_F01_3D','RFC_F04_3D','RFC_F05_3D','RFC_F06_3D','RFC_F07_3D','RFC_F08_3D','RFC_F09_3D','RFC_F10_3D','RFC_F11_3D','RFC_F12_3D','RFC_F14_3D','RFC_F15_3D','RFC_F19_3D','RFC_F20_3D','RFC_F26_3D','RFC_F28_3D','RFC_F29_3D','RFC_F30_3D','RFC_F31_3D','RFC_F32_3D','RFC_F34_3D','RFC_F35_3D','RFC_F36_3D','RFC_F37_3D','RFC_F38_3D','RFC_F40_3D','RFC_F41_3D','RFC_F43_3D','RFC_F48_3D','RFC_F52_3D','RFC_F53_3D','RFC_F54_3D']
opt_gsig = [4,4,5,4,4,5,4,4,5,5,5,5,5,4,5,4,4,4,4,4,5,5,5,4,4,4,5,4,5,4,5,5]
min_corr = [0.95,0.9,0.9,0.9,0.85,0.95,0.85,0.85,0.9,0.9,0.9,0.9,0.95,0.9,0.95,0.9,0.9,0.9,0.9,0.9,0.95,0.95,0.95,0.9,0.9,0.9,0.9,0.9,0.95,0.9,0.9,0.9]
min_pnr = [7,7,7,10,5,7,5,7,5,7,7,5,7,7,15,10,7,10,7,10,10,10,10,7,7,7,10,10,10,5,10,10]
'''

session_name = ['BOF_H02_1T','BOF_H03_1T','BOF_H04_1T','BOF_H06_1T','BOF_H07_1T','BOF_H10_1T','BOF_H11_1T','BOF_H12_1T','BOF_H13_1T','BOF_H14_1T','BOF_H15_1T','BOF_H16_1T','BOF_H17_1T','BOF_H19_1T','BOF_H22_1T','BOF_H26_1T','BOF_H27_1T','BOF_H31_1T','BOF_H32_1T','BOF_H33_1T','BOF_H39_1T']
opt_gsig = [4,5,4,5,5,5,5,5,4,5,4,5,5,4,5,5,5,4,4,4,4]
min_corr = [0.85,0.9,0.9,0.9,0.95,0.9,0.9,0.9,0.85,0.9,0.85,0.9,0.87,0.88,0.9,0.9,0.95,0.9,0.9,0.9,0.9]
min_pnr = [5,6,7,7,7,8,7,8,6,7,6,6,6,8,7,8,9,8,8,8,8]

print(len(session_name), len(opt_gsig), len(min_corr), len(min_pnr))
for i, name in enumerate(session_name):
    gSiz = opt_gsig[i]*4+1
    cnmf_dict= {'fr': 30,                   # frame rate, frames per second (NOW RECALCULATED FOR EACH FILE FROM TIMESTAMP DATA)
                'decay_time': 1,            # typical duration of calcium transient 
                'method_init': 'corr_pnr',  # use this for 1 photon
                'K': None,                  # upper bound on number of components per patch, in general None
                'gSig': (opt_gsig[i], opt_gsig[i]),             # gaussian HALF-width of a 2D gaussian kernel (in pixels), which approximates a neuron
                'gSiz': (gSiz, gSiz),           # maximal radius of a neuron in pixels
                'merge_thr': 0.8,          # merging threshold, max correlation allowed
                'p': 1,                     # order of the autoregressive system
                'tsub': 1,                  # downsampling factor in time for initialization
                'ssub': 1,                  # downsampling factor in space for initialization
                'rf': 40,                   # half-size of the patches in pixels. e.g., if rf=40, patches are 80x80
                'stride': 25,               # amount of overlap between the patches in pixels(keep it at least large as gSiz, i.e 4 times the neuron size gSig) 
                'only_init': True,          # set it to True to run CNMF-E
                'nb': 0,                    # number of background components (rank) if positive, else exact ring model with following settings: nb= 0: Return background as b and W, gnb=-1: Return full rank background B, gnb<-1: Don't return background
                'nb_patch': 0,              # number of background components (rank) per patch if nb>0, else it is set automatically
                'method_deconvolution': 'oasis',       # could use 'cvxpy' alternatively
                'low_rank_background': None,           # None leaves background of each patch intact, True performs global low-rank approximation if gnb>0
                'update_background_components': True,  # sometimes setting to False improve the results
                'min_corr': min_corr[i],                        # min peak value from correlation image
                'min_pnr': min_pnr[i],                         # min peak to noise ratio from PNR image
                'normalize_init': False,               # just leave as is
                'center_psf': True,                    # leave as is for 1 photon
                'ssub_B': 2,                           # additional downsampling factor in space for background
                'ring_size_factor': 1.5,               # radius of ring is gSiz*ring_size_factor
                'del_duplicates': True,                # whether to remove duplicates from initialization
                'border_pix': 5,                       # number of pixels to not consider in the borders
                'min_SNR': 2.5,                          # adaptive way to set threshold on the transient size
                'rval_thr': 0.95,                      # threshold on space consistency           
                'use_cnn': False}                      # whether to use CNNs for event detection  

    session_config_path = get_session_config_path(session_name[i])
    cnmf_to_config = {'cnmf_params': cnmf_dict}
    update_config(cnmf_to_config, cpath=session_config_path)

<h2>Module 3</h2>
Batch cnmf.<br/>All _MC.tif files in the root folder are to be automatically processed with CaImAn routine [Giovanucci et al., 2019] with the parameters below. Main parameters are gSig and gSiz for cell augmentation, then min_SNR as traces quality threshold. At the end, _estimates.pickle files are to be produced in the root folder. 

In [ ]:
import time
#fnames = glob(os.path.join(CONFIG['ROOT'], '*_MC.tif'))
#OR, alternatively, you can use manual file selection:
fnames = askopenfilenames(title = 'Select files for batch cnmf', initialdir = CONFIG['ROOT'], filetypes = [('TIFF files', '.tif')])

for name in tqdm.tqdm(fnames):
    fps = get_fps_from_timestamps(name[:-4-6], default_fps=20, verbose=False)
    print('timestamps average FPS: ', fps)
    session_config_path = get_session_config_path(name[-20:-10])
    cnmf_config = read_config(name=session_config_path)
    cnmf_dict = cnmf_config['cnmf_params']
    cnmf_dict.update({'fr': fps/2})
    cnmf_dict.update({'tsub': 2})

    print(f"gsig: {cnmf_dict['gSig'][1]}, mincorr: {cnmf_dict['min_corr']}, minpnr: {cnmf_dict['min_pnr']}")
    out_name = name[:-4-6] + f'_estimates.pickle'
    print('estimate output name: ', out_name)
    DoCNMF(name,
           cnmf_dict,
           out_name=out_name,
           verbose=False,
           noise_ampl=1e-5)
    
    #CleanMemmaps(name)  
    temp_pathway = CONFIG['TEMP_PATHWAY']
    sub_name = '/'.join(name.split('/')[6:])
    memmap_name = f'{temp_pathway}/{sub_name}'
    
    err_cnt = 0
    while err_cnt < 100:
        try:
            CleanMemmaps(memmap_name)
            break
        except PermissionError:
            time.sleep(1)
            err_cnt += 1
    print('CleanMemmaps attemps:', err_cnt)

<h2>Module 4</h2>
User inspection of cnmf results.<br/>
At this stage, previously saved timestamps are to be merged with cnmf results.

Bokeh server configuration for interactive visualization. Set the port to match your Jupyter Notebook server.

In [ ]:
import os
os.environ['BOKEH_ALLOW_WS_ORIGIN'] = 'localhost:8888'  # Укажите порт вашего Jupyter Notebook, например, 8888

Batch compression of estimates files. Reduces file size by removing large matrices and optionally bad components.

In [2]:
from tkinter.filedialog import askopenfilenames
from pathlib import Path
from estimates_compression import compress_estimates_ultra_lightweight
import pickle
import time

# Interactive file selection
pickle_files = askopenfilenames(
    title='Select estimates files to compress',
    initialdir=CONFIG['ROOT'],
    filetypes=[('Pickle files', '*.pickle')]
)

if not pickle_files:
    print("No files selected. Aborting.")
else:
    pickle_files = [Path(f) for f in pickle_files]
    
    # Destination directory (coded pattern based on first file's parent)
    source_dir = pickle_files[0].parent
    dest_dir = source_dir.parent / f"{source_dir.name}_ultra_lightweight"

    print(f"\nSource: {source_dir}")
    print(f"Destination: {dest_dir}")
    print(f"Selected {len(pickle_files)} files")

    # Check if destination exists and warn
    if dest_dir.exists():
        print(f"\nWARNING: Destination directory already exists!")
        print(f"Files may be overwritten. Continue? (type 'yes' to proceed)")
        response = input().strip().lower()
        if response != 'yes':
            print("Aborted by user.")
            raise StopIteration
    else:
        # Create destination
        dest_dir.mkdir(parents=True, exist_ok=True)
        print(f"Created destination directory: {dest_dir}")

    print(f"\nProcessing {len(pickle_files)} files")
    print("=" * 80)

    total_original = 0
    total_compressed = 0
    successful = 0
    failed = 0
    start_time = time.time()

    # Process with progress bar
    for i, input_path in enumerate(tqdm.tqdm(pickle_files, desc="Compressing"), 1):
        output_path = dest_dir / input_path.name

        try:
            # Load estimates
            with open(input_path, 'rb') as f:
                est = pickle.load(f)

            original_size = input_path.stat().st_size / (1024**2)

            # Compress (remove bad components by default)
            est_compressed, savings, total_saved = compress_estimates_ultra_lightweight(
                est,
                remove_bad_components=True
            )

            # Save compressed
            with open(output_path, 'wb') as f:
                pickle.dump(est_compressed, f, protocol=pickle.HIGHEST_PROTOCOL)

            new_size = output_path.stat().st_size / (1024**2)
            compression_ratio = (1 - new_size / original_size) * 100

            total_original += original_size
            total_compressed += new_size
            successful += 1

            tqdm.tqdm.write(f"  [{i}/{len(pickle_files)}] {input_path.name}: "
                           f"{original_size:.1f}MB -> {new_size:.1f}MB "
                           f"({compression_ratio:.1f}% saved)")

        except Exception as e:
            failed += 1
            tqdm.tqdm.write(f"  [FAILED] {input_path.name}: {e}")
            total_original += input_path.stat().st_size / (1024**2)
            total_compressed += input_path.stat().st_size / (1024**2)

    # Summary
    elapsed = time.time() - start_time
    print("\n" + "=" * 80)
    print("BATCH COMPRESSION COMPLETE")
    print("-" * 80)
    print(f"  Total files: {len(pickle_files)}")
    print(f"  Successful: {successful}")
    print(f"  Failed: {failed}")
    print(f"  Time elapsed: {elapsed/60:.1f} minutes")
    print(f"  Original size: {total_original:.2f} MB ({total_original/1024:.2f} GB)")
    print(f"  Compressed size: {total_compressed:.2f} MB ({total_compressed/1024:.2f} GB)")
    print(f"  Total saved: {total_original - total_compressed:.2f} MB")
    print(f"  Compression ratio: {(1 - total_compressed/total_original)*100:.1f}%")
    print("=" * 80)


Source: C:\Users\User\PycharmProjects\bearmind\output\inspection_artifacts_LNOF_J12_1D
Destination: C:\Users\User\PycharmProjects\bearmind\output\inspection_artifacts_LNOF_J12_1D_ultra_lightweight
Selected 1 files
Created destination directory: C:\Users\User\PycharmProjects\bearmind\output\inspection_artifacts_LNOF_J12_1D_ultra_lightweight

Processing 1 files


  [1/1] LNOF_J12_1D_processed.pickle: 604.0MB -> 604.0MB (0.0% saved)


Compressing: 100%|██████████| 1/1 [00:00<00:00,  1.16it/s]



BATCH COMPRESSION COMPLETE
--------------------------------------------------------------------------------
  Total files: 1
  Successful: 1
  Failed: 0
  Time elapsed: 0.0 minutes
  Original size: 604.05 MB (0.59 GB)
  Compressed size: 604.05 MB (0.59 GB)
  Total saved: 0.00 MB
  Compression ratio: 0.0%


Batch autoinspection for processing multiple estimates files with automatic quality control using ML models and rule-based filtering.

<h3>Capcan autoinspection</h3>

In [ ]:
from ae_launch import run_auto_inspection

def run_single_autoinspection(fname, verbose=True):
    """
    Run autoinspection on a single estimates file.

    Args:
        fname: Path to estimates pickle file
        verbose: Print detailed output

    Returns:
        dict: Result from run_auto_inspection, or None if failed
    """
    # Define custom deletion rules (or use None for defaults)
    deletion_rules = None  # Use default rules

    try:
        result = run_auto_inspection(
            fname,
            fps=None,

            # --- Session naming ---
            session_name = None,

            # --- Metrics extraction parameters ---
            comps_to_select = None,
            cthr = 0.35,
            corr_thr = 0.6,
            num_sessions = 1,
            match_threshold = 3,
            sf = None,
            ef = None,
            ds = 1,
            include_event_based = True,
            include_heavy = True,
            detect_corner_artifacts = True,
            corner_artifact_params = None,
            event_method = 'wavelet',
            n_iter = 3,
            correlation_method = 'pearson',
            wavelet_backend = 'auto',  # GPU if available, else CPU
            hybrid_kinetics = True,

            # --- Brain selection ---
            brain = 'hybrid',
            ml_model_path = "production_models/ebm_v9_iter8.pkl",
            ml_threshold = 0.72,

            # --- Decision parameters ---
            deletion_rules = deletion_rules,
            pxlthr_distance_boundary = 5,
            d_snr_thr = 10,
            enable_merge = True,

            # --- Tracking ---
            track_criteria_failures = True,

            # --- Artifact saving ---
            save_artifacts = True,
            artifacts_path = './output',
            save_estimates = True,
            save_matrices = True,
            save_corner_detection = True,
            compress_estimates = True,

            # --- Verbosity ---
            verbose = verbose
        )

        est = result['estimates']
        if verbose:
            print(f"SUCCESS: {len(est.idx_components)} neurons processed")

        return result

    except Exception as e:
        print(f"FAILED: {fname}")
        print(f"  Error: {e}")
        return None


# Single file autoinspection (interactive)
fname = askopenfilename(title = 'Select estimates file for examination',
                        initialdir = CONFIG['ROOT'],
                        filetypes = [('estimates files', '*estimates.pickle')])

if fname:
    result = run_single_autoinspection(fname, verbose=True)
    if result:
        est = result['estimates']
        print(f"Total: {len(est.idx_components)} neurons processed")

In [5]:
from tkinter.filedialog import askopenfilenames
from pathlib import Path
import time

# BATCH AUTOINSPECTION
estimates_files = askopenfilenames(
    title='Select estimates files for autoinspection',
    initialdir=CONFIG['ROOT'],
    filetypes=[('Estimates files', '*estimates.pickle')]
)

if not estimates_files:
    print("No files selected. Aborting.")
else:
    estimates_files = [Path(f) for f in estimates_files]
    
    print(f"\nFound {len(estimates_files)} files for autoinspection")
    print("=" * 80)

    # Tracking
    successful = 0
    failed = 0
    failed_files = []
    consecutive_failures = 0
    MAX_CONSECUTIVE_FAILURES = 3

    start_time = time.time()

    # Process with progress bar
    for i, fpath in enumerate(tqdm.tqdm(estimates_files, desc="Autoinspecting"), 1):
        fname = str(fpath)

        tqdm.tqdm.write(f"\n[{i}/{len(estimates_files)}] Processing: {fpath.name}")

        # Run autoinspection
        result = run_single_autoinspection(fname, verbose=False)

        if result is not None:
            # Success
            est = result['estimates']
            successful += 1
            consecutive_failures = 0  # Reset counter
            tqdm.tqdm.write(f"  [OK] {len(est.idx_components)} neurons kept")
        else:
            # Failure
            failed += 1
            failed_files.append(fpath.name)
            consecutive_failures += 1
            tqdm.tqdm.write(f"  [FAILED] See error above")

            # Check for consecutive failure threshold
            if consecutive_failures >= MAX_CONSECUTIVE_FAILURES:
                print("\n" + "=" * 80)
                print("!" * 80)
                print("CRITICAL WARNING: 3 CONSECUTIVE FAILURES DETECTED")
                print("!" * 80)
                print(f"  Failed files: {failed_files[-3:]}")
                print(f"  This may indicate:")
                print(f"    - Corrupt estimates files")
                print(f"    - Missing dependencies")
                print(f"    - Configuration errors")
                print(f"    - Memory issues")
                print()
                print("  STOPPING BATCH PROCESSING FOR SAFETY")
                print("  Please investigate the errors above before continuing.")
                print("=" * 80)
                break

    # Final summary
    elapsed = time.time() - start_time
    print("\n" + "=" * 80)
    print("BATCH AUTOINSPECTION COMPLETE")
    print("-" * 80)
    print(f"  Total files: {len(estimates_files)}")
    print(f"  Processed: {i}")
    print(f"  Successful: {successful}")
    print(f"  Failed: {failed}")
    print(f"  Time elapsed: {elapsed/60:.1f} minutes")
    print(f"  Average time per file: {elapsed/max(i,1):.1f} seconds")

    if failed > 0:
        print(f"\n  Failed files:")
        for fname in failed_files:
            print(f"    - {fname}")

    print("=" * 80)


Found 3 files for autoinspection



[1/3] Processing: NOF_H03_2D_gsig4_mincorr0.92_minpnr7_estimates.pickle


Autoinspecting:   0%|          | 0/3 [00:00<?, ?it/s]

C:\Users\User\PycharmProjects\bearmind\data\raw_compressed\NOF_H03_2D_
[]
[1/4] Preparing traces for 174 neurons...
[2/4] Computing correlation matrix (spearman)...
[3/4] Extracting spatial metrics...
Computing boundary distances...



174it [00:00, 20081.14it/s]


...


272it [00:00, 4709.18it/s]


[4/4] Computing wavelet event-based metrics (this may take a while)...
      Event metrics completed in 186.18s (0.684s/neuron)
      Cached 270 reconstructions
Edge artifact detection: 20 artifacts (7.4%) [corner:0, ellipse:12, both:8]
Metrics extraction completed: 272 neurons in 238.0s
Saved edge artifact visualization to output\inspection_artifacts_NOF_H03_4D_gsig4_mincorr0.92_minpnr7\edge_artifacts.png
[save_processed_estimates] Compressed estimates, saved 12.5 MB
[save_processed_estimates] Saving estimates WITH cached metrics (272 rows)


  [OK] 272 neurons kept


Autoinspecting: 100%|██████████| 3/3 [10:18<00:00, 206.25s/it]



BATCH AUTOINSPECTION COMPLETE
--------------------------------------------------------------------------------
  Total files: 3
  Processed: 3
  Successful: 3
  Failed: 0
  Time elapsed: 10.3 minutes
  Average time per file: 206.2 seconds


<h3>Manual inspection</h3>

In [ ]:
import time

fname = askopenfilename(title = 'Select estimates file for examination',
                        initialdir = CONFIG['ROOT'],
                        filetypes = [('estimates files', '*.pickle')])

bkapp_kwargs = {
    'mode': 'capcan',          # operation mode, can be 'legacy'/'capcan'
    'start_frame': 0,          # start from this frame
    'end_frame': 90000,        # end at this frame
    'num_sessions': 1,         # for merged multi-session recordings
    'match_threshold': 1,      # threshold number of sessions with significant correlation, takes effect for merged multi-session recordings
    'include_event_based': 1,  # compute event-based metrics or not. Takes time (~0.2 s per neuron, a minute or two for typical exp size)
    'include_heavy': 1,        # compute heavy reconstruction-based metrics or not (may take about 5-10 minutes)
    'event_method': 'wavelet', # event detection method
    'n_iter': 3,               # number of iterations for iterative reconstruction
    'color_by_ml_probability': True,
    'ml_threshold': 0.72,
    'ml_model_path': 'production_models/ebm_v9_iter8.pkl',  # Path to ML model for neuron classification
    'detect_corner_artifacts': True,   # Enable/disable corner artifact detection
    'corner_artifact_params': None,    # Custom params for corner detection (None = defaults)
    'correlation_method': 'pearson',   # 'pearson' or 'spearman' for correlation computation
    'downsampling': 5,         # take every 'ds' frame
    'fill_alpha': 0.8,         # selected neuron transparency
    'ns_alpha': 0.2,           # non-selected neuron transparency
    'line_width': 0.5,         # border width
    'cthr': 0.35,              # coutour_thr from caiman (% of signal inside a patch), affects patch size
    'sort_order': 'up',        # sorting order
    'corr_thr': 0.35,          # threshold for truncated corr matrix
    'line_alpha': 0.5,         # border transparency
    'trace_line_width': 1,     # trace line width
    'trace_alpha': 0.7,        # trace transparency
    'size': 450,               # left/right widget size
    'metrics_width': 200,      # central metrics widget width in pixels
    'button_width': 50,        # button width in pixels
    'compress_estimates': True,  # compress estimates before saving (reduces file size)
    'verbose': 0,
    'enable_gpu_backend': 1,
    'oh_shit': 0
}

ExamineCells(fname, default_fps=30, bkapp_kwargs=bkapp_kwargs)

### Batch export data and metadata
Export calcium traces, amplitude spikes (ASP), and reconstructions to `.npz` files with corresponding `.json` metadata. Automatically incorporates feedback corrections if a feedback file is present in the same folder.

In [ ]:
# Batch export data and metadata from processed estimates
from pathlib import Path
from tkinter.filedialog import askopenfilenames
import numpy as np
from export_estimates_data import (
    load_estimates, extract_session_name, get_fps,
    find_feedback_file, load_feedback, apply_feedback,
    extract_data, build_metadata, export, export_filters_mat,
    DEFAULT_ML_THRESHOLD
)

# Multi-select processed estimates files
pickle_files = askopenfilenames(
    title='Select processed estimates files for export',
    initialdir=CONFIG['ROOT'],
    filetypes=[('Processed estimates', '*_processed.pickle')]
)

if not pickle_files:
    print("No files selected.")
else:
    pickle_files = [Path(f) for f in pickle_files]
    print(f"Selected {len(pickle_files)} files for export")
    print("=" * 60)

    successful = 0
    failed = 0

    for i, est_path in enumerate(pickle_files, 1):
        folder = est_path.parent
        print(f"\n[{i}/{len(pickle_files)}] {est_path.name}")

        try:
            # Load estimates
            est = load_estimates(est_path)
            session_name = extract_session_name(est)
            fps = get_fps(session_name)
            print(f"  Session: {session_name}, FPS: {fps}")

            # Get component indices and apply ML threshold filter
            component_indices = est.idx_components.copy()
            n_before_ml_filter = len(component_indices)
            ml_filtered = False
            ml_threshold_used = None

            # Apply ML threshold filtering if metrics available
            if hasattr(est, 'metrics_df') and est.metrics_df is not None:
                df = est.metrics_df
                if 'ml_keep_probability' in df.columns:
                    # Get threshold from config or use default
                    threshold = DEFAULT_ML_THRESHOLD
                    if hasattr(est, 'autoinspection_config') and est.autoinspection_config:
                        threshold = est.autoinspection_config.get('ml_threshold', DEFAULT_ML_THRESHOLD)

                    ml_approved_mask = df['ml_keep_probability'] >= threshold
                    ml_approved_indices = set(df.loc[ml_approved_mask, 'component_idx'].tolist())
                    component_indices = np.array([i for i in component_indices if i in ml_approved_indices])

                    ml_filtered = True
                    ml_threshold_used = threshold
                    print(f"  ML filter ({threshold}): {n_before_ml_filter} -> {len(component_indices)}")

            # Check for feedback and apply AFTER ML filter
            feedback_path = find_feedback_file(est_path, session_name)
            feedback_applied = False
            feedback_summary = None

            if feedback_path:
                print(f"  Found feedback: {feedback_path.name}")
                feedback_df = load_feedback(feedback_path)
                component_indices, feedback_summary = apply_feedback(component_indices, feedback_df)
                feedback_applied = True
                print(f"  Applied: {feedback_summary['n_fp_removed']} FP removed, "
                      f"{feedback_summary['n_fn_added']} FN added")
            else:
                print(f"  No feedback file found")

            # Build ML filter info for metadata
            ml_filter_info = {
                'ml_filtered': ml_filtered,
                'threshold_used': ml_threshold_used,
                'n_before': n_before_ml_filter,
                'n_after': len(component_indices) if ml_filtered else None
            }

            # Extract and export
            data = extract_data(est, component_indices)
            metadata = build_metadata(est, fps, session_name, feedback_applied, feedback_summary, ml_filter_info)
            npz_path, json_path = export(data, metadata, session_name, folder)

            # Export spatial filters
            mat_path = folder / f'{session_name}_filters.mat'
            export_filters_mat(est, component_indices, mat_path)

            print(f"  Exported: {len(component_indices)} components")
            successful += 1

        except Exception as e:
            print(f"  [ERROR] {e}")
            import traceback
            traceback.print_exc()
            failed += 1

    print("\n" + "=" * 60)
    print(f"BATCH EXPORT COMPLETE: {successful} succeeded, {failed} failed")

Manual seed placement for neurons that were missed during automatic detection.

In [ ]:
fnames = askopenfilenames(title = 'Select files for batch cnmf',
                          initialdir = CONFIG['ROOT'],
                          filetypes = [('TIFF files', '.tif')])

#fnames = glob(os.path.join(CONFIG['ROOT'], '*_MC.tif'))

ManualSeeds(fnames[0], size=800, cnmf_dict=None)

Redo cnmf with manually added seeds (optional).<br/>
NB!! By running the cell below, you will rewrite existing estimates files!!<br/>
Then you can return to the section above and inspect the rewritten estimates.

In [ ]:
s_names = glob(os.path.join(CONFIG['ROOT'], '*seeds.pickle'))
#OR, alternatively, you can use manual file selection:
#s_names = askopenfilenames(title = 'Select seeds files for re-CNMFing', initialdir = CONFIG['ROOT'], filetypes = [('seeds files', '*seeds.pickle')])


for s_name in s_names:
    base_name = s_name.partition('_seeds')[0][:-4]
    
    e_names = glob(base_name + '_estimates.pickle')
    tif_names = glob(base_name + '.tif')
    ReDoCNMF(s_name, e_name=None, tif_name=tif_names[0], cnmf_dict=cnmf_dict)
    CleanMemmaps(base_name)

<h2>Module 5</h2>
Batch event detection. <br/>
INPUT: (timestamped) cnmf raw traces as *_traces.csv files<br/>
OUTPUT: detected events as *_spikes.csv files; pickles with events (cell-wise list of event-wise lists with dictionaries) and also, interactive .html plot with traces and events.

In [ ]:
fnames = glob(CONFIG['ROOT'] + '*traces.csv')
#OR, alternatively, you can use manual file selection:
#fnames = askopenfilenames(title = 'Select traces for event detection', initialdir = CONFIG['ROOT'], filetypes = [('traces files', '*traces.csv')])

sd_dict = {'thr': 4,        #threshold for peaks in Median Absolute Deviations (MADs)                   
           'sigma' : 7,     #smoothing parameter for peak detection, frames
           'est_ton' : 0.5, #estimated event rising time, s
           'est_toff' : 2,  #estimated event decay time, s
           'draw_details': True} #whether to draw smoothed traces, peaks, pits and fits 

for name in fnames:
    FitEvents(name, opts = sd_dict)


Also, just in case, you may draw existed pairs of traces and spikes right here:

In [ ]:
fnames = glob(CONFIG['ROOT'] + '*traces.csv')
for name in fnames:
    DrawSpEvents(name, name.replace('traces','spikes'))


<h1> Wavelet event detection</h1>

In [ ]:
import scipy
print(scipy.__version__)


In [ ]:
def read_traces(fname):
    trdata = pd.read_csv(fname)
    time = trdata['time_s'].values
    traces = np.array(trdata)[:,1:].T
    return traces, time

#fnames = glob(CONFIG['ROOT'] + '*traces.csv')
fnames = askopenfilenames(title = 'Select traces for event detection', initialdir = CONFIG['ROOT'], filetypes = [('traces files', '*traces.csv')])

wvt_param_dict = {'fps': 30,        # fps, frames                   
                  'sigma' : 8,      # smoothing parameter for peak detection, frames
                  'beta' : 2,       # Generalized Morse Wavelet parameter, FIXED
                  'gamma' : 3,      # Generalized Morse Wavelet parameter, FIXED
                  'eps': 10,         # spacing beween consecutive events, frames
                  'manual_scales': np.logspace(2.5,5.5,50, base=2),

                  # ridge filtering params
                  'scale_length_thr': 40,  # min number of scales where ridge is present thr, higher = less events. max=len(manual_scales)
                  'max_scale_thr': 7,      # index of a scale with max ridge intensity thr, higher = less events. < 5 = noise, > 20 = huge events
                  'max_ampl_thr': 0.05,    # max ridge intensity thr, higher = less events. < 5 = noise, > 20 = huge events
                  'max_dur_thr': 100,      # max event duration thr, higher = more events (but probably strange ones)
}


for fname in fnames:
    traces, time = read_traces(fname)
    st_evinds, end_evinds, all_ridges = extract_wvt_events(traces, wvt_param_dict)
    events_to_csv(time, st_evinds, end_evinds, fname)

Recompute with different filtering params:

In [ ]:
wvt_param_dict['scale_length_thr'] = 40,  # min number of scales where ridge is present thr, higher = less events. max=len(manual_scales)
wvt_param_dict['max_scale_thr'] = 7,      # index of a scale with max ridge intensity thr, higher = less events. < 5 = noise, > 20 = huge events
wvt_param_dict['max_ampl_thr'] = 0.05,    # max ridge intensity thr, higher = less events. < 5 = noise, > 20 = huge events
wvt_param_dict['max_dur_thr'] = 200,      # max event duration thr, higher = more events (but probably strange ones)

events = []
for i in range(traces.shape[0]):
    st_evinds, end_evinds = get_events_from_ridges(all_ridges[i],
                                                   scale_length_thr=40,
                                                   max_scale_thr=7,
                                                   max_ampl_thr=0.05,
                                                   max_dur_thr=200)

    events.append(end_evinds)

In [ ]:
st = 0
end = 10000
neuron_ind = 10

sig = gaussian_filter1d(traces[neuron_ind], sigma=wvt_param_dict['sigma'])
sig = traces[neuron_ind]

fig, ax = plt.subplots(figsize=(10,8))
ax.set_xlim(st, end)
ax.plot(np.arange(st, end), sig[st:end], c='b')


#for ev in end_evinds[neuron_ind]:
#    ax.axvline(ev, c='r', alpha=0.5)

for ev in events[neuron_ind]:
    ax.axvline(ev, c='r', alpha=0.5)